# ISOM 839 · Session 7 — Integer Optimization II: Scheduling, Location & the Midterm Case

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Binary variables meet the workforce. Set covering is the pattern behind fire stations, delivery zones and every staff roster: *cover all the demand with the fewest, or cheapest, resources.* Tonight you meet a second solver built for exactly this logic — Google's **CP-SAT** — and then the main event: the midterm case launches, and the data is in this notebook.

In [ ]:
%pip install -q ortools gurobipy pandas
from ortools.sat.python import cp_model
import pandas as pd
print('CP-SAT ready')

---
## Part 1 — Barista shift scheduling with CP-SAT (tonight's lab)

The campus café needs a week of shifts covered: **open, mid, close** on Monday through Friday, each with a minimum headcount. Six baristas, different wages, different availability. Rules: nobody works more than 5 shifts a week, and nobody closes and then opens the next morning (the "clopen").

In CP-SAT, variables are integers or booleans, constraints are written as plain Python arithmetic and logic, and the solver is a constraint-programming engine rather than branch & bound over an LP. Same decisions, different machine.

In [ ]:
baristas = ['Ana', 'Ben', 'Chen', 'Dara', 'Eli', 'Fay']
wage     = {'Ana': 18, 'Ben': 16, 'Chen': 17, 'Dara': 19, 'Eli': 16, 'Fay': 18}
days     = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
blocks   = {'open': 4, 'mid': 4, 'close': 4}            # hours per block
shifts   = [(d, b) for d in days for b in blocks]
need     = {'open': 2, 'mid': 1, 'close': 2}             # minimum baristas per block

unavailable = {                                          # who cannot work what
    'Ana':  [('Fri', 'close')],
    'Ben':  [('Mon', 'open'), ('Tue', 'open')],
    'Chen': [('Wed', 'mid'), ('Wed', 'close')],
    'Dara': [('Thu', 'open'), ('Fri', 'open')],
    'Eli':  [],
    'Fay':  [('Mon', 'close'), ('Tue', 'close'), ('Wed', 'close')],
}

def build(need):
    model = cp_model.CpModel()
    assign = {(b, s): model.NewBoolVar(f'{b}_{s[0]}_{s[1]}') for b in baristas for s in shifts}
    for s in shifts:                                                    # cover every block
        model.Add(sum(assign[b, s] for b in baristas) >= need[s[1]])
    for b in baristas:
        for s in unavailable[b]:                                        # availability
            model.Add(assign[b, s] == 0)
        model.Add(sum(assign[b, s] for s in shifts) <= 5)               # max 5 shifts
        for d1, d2 in zip(days, days[1:]):                              # no clopen
            model.AddBoolOr([assign[b, (d1, 'close')].Not(), assign[b, (d2, 'open')].Not()])
    model.Minimize(sum(wage[b] * blocks[s[1]] * assign[b, s] for b in baristas for s in shifts))
    return model, assign

def solve_and_print(need):
    model, assign = build(need)
    solver = cp_model.CpSolver(); solver.parameters.max_time_in_seconds = 10
    status = solver.Solve(model)
    print('status:', solver.StatusName(status), f'· weekly labor ${solver.ObjectiveValue():,.0f} · {solver.WallTime():.3f}s')
    for d in days:
        row = []
        for blk in blocks:
            who = [b for b in baristas if solver.Value(assign[b, (d, blk)])]
            row.append(f'{blk}: {", ".join(who):18s}')
        print(f'  {d}  ' + ' | '.join(row))
    print('  shifts per barista:', {b: sum(solver.Value(assign[b, s]) for s in shifts) for b in baristas})
    return solver, assign

solver, assign = solve_and_print(need)

Read the roster. Five baristas work the full five shifts; **Dara, the most expensive at $19, never works** — the solver would rather max out everyone else. Every block is covered, nobody clopens, and CP-SAT proved it optimal in milliseconds. (Is that roster *fair*? No. Fairness is a constraint you have not written yet — try `>= 2` shifts per person and see what it costs.)

**Step 4 — re-plan under a demand spike.** Friday evening now needs 3 baristas. Change one number and re-solve. Who absorbed the extra shift, and what did it cost?

In [ ]:
# Friday close needs 3 — add one constraint on top of the base model and re-solve
model, assign = build(need)
model.Add(sum(assign[b, ('Fri', 'close')] for b in baristas) >= 3)
solver = cp_model.CpSolver(); solver.Solve(model)
print(f'With a Friday-evening spike: ${solver.ObjectiveValue():,.0f} per week')
print('  Friday close:', [b for b in baristas if solver.Value(assign[b, ('Fri', 'close')])])

### 💬 Discussion
Your café has 6 people and 15 blocks. ECCO's replenishment system makes ~300,000 decisions a month across 536 stores — the same class of model at industrial scale, several million euros a year in savings. As you open the case below, ask the ECCO question: **which decisions are truly yes/no, and which are quantities?**

---
# 🏆 The Midterm Case — Beacon Outfitters: Four Weeks of Replenishment

**Full brief, deliverables and rubric: [isom-839.vercel.app/case](https://isom-839.vercel.app/case).** Teams on Canvas (People → Groups → Teams). Due date on Canvas.

Beacon Outfitters runs **12 stores** across New England from one distribution center in Worcester. Every week the DC ships product to the stores; every store has a shelf capacity, a truck that costs money each time it visits, and a demand forecast for **8 SKUs** over the next **4 weeks** — with a cold snap coming in weeks 3–4. Product ships in **case packs** (you cannot send half a case), the DC's supply of parkas, beanies and gloves is tight, and the merchandising VP wants to know: *what do we ship, where, when — and what is it worth?*

The data is five small CSV files. Load them straight from the course repo:

In [ ]:
RAW = 'https://raw.githubusercontent.com/harslan/isom-839/master/public/session-07/case/'
stores = pd.read_csv(RAW + 'stores.csv')
skus   = pd.read_csv(RAW + 'skus.csv')
demand = pd.read_csv(RAW + 'demand_forecast.csv')
inv0   = pd.read_csv(RAW + 'opening_inventory.csv')
dc     = pd.read_csv(RAW + 'dc_availability.csv')
print(stores.to_string(index=False)); print(); print(skus.to_string(index=False)); print()
print('demand rows:', len(demand), '· opening inventory rows:', len(inv0), '· DC availability rows:', len(dc))

**Before Session 8 (the check-in):** summary statistics and a one-page model outline. Start here.

In [ ]:
# Summary statistics — complete these for the Session 8 check-in
by_week = demand.groupby('week')['forecast_units'].sum()
print('Forecast demand by week:\n', by_week.to_string())
by_sku  = demand.groupby('sku')['forecast_units'].sum().rename('forecast')
supply  = dc.groupby('sku')['units_available'].sum().rename('dc_supply')
opening = inv0.groupby('sku')['on_hand_units'].sum().rename('opening_stock')
print('\nSupply vs demand by SKU:\n', pd.concat([by_sku, supply, opening], axis=1).assign(gap=lambda t: t.dc_supply + t.opening_stock - t.forecast).to_string())
print('\nTotal store shelf capacity:', stores.shelf_capacity_units.sum(), '· total opening stock:', inv0.on_hand_units.sum())

# TODO: which SKUs are short? which stores have the least shelf room relative to their 4-week demand?
# TODO: what does a truck visit cost per unit delivered at the smallest and largest store?

### Your model outline (bring one page to Session 8)

Decide, for every **store × SKU × week**: how many **cases** to ship. Everything else follows from that.

- **Decision variables** — cases shipped (integer); units sold, lost and held (continuous); whether a truck visits a store in a week (binary).
- **Objective** — maximize four-week profit: margin on units sold, minus holding cost on units left on the shelf each week, minus truck visits.
- **Constraint families (at least four)** — inventory flow at every store/SKU/week; sales cannot exceed forecast; shelf capacity after delivery; truck capacity, switched on by the visit variable (the fixed-charge link from Session 6); DC availability per SKU, cumulative across weeks (unused supply carries forward).
- **Integers** — cases ship in case packs. `units = case_pack × cases`, with `cases` integer. That one line is why this is an integer program.

Sanity check before the check-in: with no shipments at all, Beacon still sells its opening stock. Your model's optimum should beat that by a wide margin — and the gap *is* the business case. What the VP will ask: which SKU is the real bottleneck, and what would 100 more units of it at the DC be worth?

In [ ]:
import gurobipy as gp
from gurobipy import GRB

S = list(stores.store_id); K = list(skus.sku); W = sorted(demand.week.unique())
# helpful lookups
shelf  = dict(zip(stores.store_id, stores.shelf_capacity_units))
vcost  = dict(zip(stores.store_id, stores.delivery_cost_per_visit))
truck  = dict(zip(stores.store_id, stores.truck_capacity_units))
pack   = dict(zip(skus.sku, skus.case_pack))
margin = {r.sku: r.retail_price - r.unit_cost for r in skus.itertuples()}
hold   = dict(zip(skus.sku, skus.holding_cost_per_unit_week))
fc     = {(r.store_id, r.sku, r.week): r.forecast_units for r in demand.itertuples()}
on     = {(r.store_id, r.sku): r.on_hand_units for r in inv0.itertuples()}
avail  = {(r.sku, r.week): r.units_available for r in dc.itertuples()}

case = gp.Model('beacon_outfitters')
# TODO 1: variables — cases[s,k,w] integer; sales, lost, inv continuous; visit[s,w] binary
# TODO 2: objective — margin * sales − holding * inv − delivery cost * visit
# TODO 3: inventory flow — previous inventory (opening stock in week 1) + case_pack*cases == sales + inv
# TODO 4: demand — sales + lost == forecast
# TODO 5: shelf — inventory carried in + units delivered <= shelf capacity
# TODO 6: truck — units delivered <= truck capacity * visit   (the fixed-charge link)
# TODO 7: DC — cumulative units shipped of each SKU through week w <= cumulative availability through w
# case.optimize()
# TODO 8: report — profit, fill rate by SKU, visits by store and week, the binding SKU, and the value of 100 more units of it

---
### What to bring to Session 8
- [ ] The summary statistics cell complete, with your two TODO answers
- [ ] A one-page outline: variables, objective, at least four constraint families, in business English
- [ ] One question you want answered in the five-minute check-in

**The model is the easy part.** The memo and the five-minute briefing are where the grade lives — the best model nobody understands loses to a good model everyone does.